### Connection MYSQL

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [ ]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

### Library

In [ ]:
import pandas as pd

### AKM

In [ ]:
akmQuery=text("""
WITH students AS (
    SELECT
        registers.id,
        registers.uuid,
        registers.nim,
		registers.semester_code AS semester_student,
        prodi_registers.name AS prodi_student,
        id_jenis_keluar AS jenis_keluar_student
    FROM
        mahasiswa_registers AS registers
    JOIN 
        prodi AS prodi_registers ON registers.prodi_uuid = prodi_registers.uuid
    WHERE
        registers.deleted_at IS NULL 
        AND registers.is_external = 0 
        AND registers.is_archive = 0
),
akm AS (
    SELECT
        s.nim,
		s.semester_student,
        s.prodi_student,
        s.jenis_keluar_student,
        cuti.semester_start AS semester_inactive_start,
		cuti.semester_return AS semester_inactive_return,
        keluar.semester_code AS semester_leave,
        keluar.jeniskeluar_code,
        kelas_laporan.semester_code AS semester_class_report,
		mbkm_activities.semester_code AS semester_mbkm
    FROM students s
    -- Join Cuti
    LEFT JOIN mahasiswa_cuti AS cuti 
        ON cuti.mahasiswa_register_uuid = s.uuid 
        AND cuti.deleted_at IS NULL 
        AND cuti.latest_status = 'approved'
    -- Join Keluar
    LEFT JOIN mahasiswa_keluar_details AS keluar_details 
        ON keluar_details.mahasiswa_register_uuid = s.uuid
    LEFT JOIN mahasiswa_keluar AS keluar 
        ON keluar_details.keluar_id = keluar.id
    -- Join Nilai & Kelas Laporan
    LEFT JOIN mahasiswa_nilais AS nilai 
        ON nilai.mahasiswa_register_id = s.id 
        AND nilai.deleted_at IS NULL
    LEFT JOIN kelas 
        ON nilai.kelas_id = kelas.id 
        AND kelas.deleted_at IS NULL
    LEFT JOIN kelas_laporan 
        ON kelas.kelas_laporan_uuid = kelas_laporan.uuid  
        AND kelas_laporan.deleted_at IS NULL
		-- Join MBKM
	LEFT JOIN mbkm_mahasiswa_applications AS mbkm_mahasiswa
		ON mbkm_mahasiswa.mahasiswa_register_uuid = s.uuid
	LEFT JOIN mahasiswa_activities AS mbkm_activities
		ON mbkm_mahasiswa.mahasiswa_activity_uuid = mbkm_activities.uuid
)
SELECT
	nim,
	semester_student,
    prodi_student,
    jenis_keluar_student,
    semester_class_report,
    semester_leave,
    jeniskeluar_code,
    semester_inactive_start,
	semester_inactive_return,
	semester_mbkm
FROM
    akm
GROUP BY
	nim,
	semester_student,
    prodi_student,
    jenis_keluar_student,
    semester_class_report,
    semester_leave,
    jeniskeluar_code,
    semester_inactive_start,
	semester_inactive_return,
	semester_mbkm
""")

In [ ]:
akmDf = pd.read_sql(akmQuery, siakadEngine)

In [ ]:
akmDf.info()

In [ ]:
akmDf

### Nilai Outlier

In [ ]:
outlier = (akmDf['jenis_keluar_student'] != akmDf['jeniskeluar_code']) & (
    akmDf['jenis_keluar_student'].notna()
)

In [ ]:
akmDf.loc[outlier, ['nim', 'prodi_student', 'jenis_keluar_student', 'jeniskeluar_code']].drop_duplicates()

In [ ]:
akmClearDf = akmDf[~outlier].copy()

### Master Semester

In [ ]:
semesterQuery = text("""
SELECT
    code, year_of_study
FROM
    semesters
WHERE
    year_of_study >= ( YEAR ( CURDATE()) - 4 ) AND
     year_of_study <= ( YEAR ( CURDATE())) AND
     semester != 3
""")

semesterDf = pd.read_sql(semesterQuery, siakadEngine)

In [ ]:
semesterDf

### Unique Valid Student

In [ ]:
studentDf = akmClearDf.loc[
    (akmDf['semester_leave'].isna()) |
    (
        (akmDf['semester_leave'] >= semesterDf['code'].iloc[0]) &
        (akmDf['semester_leave'] <= semesterDf['code'].iloc[-1])
    ),
    ['nim', 'prodi_student', 'semester_leave', 'jeniskeluar_code']
].drop_duplicates()

In [ ]:
studentDf

### Cross Join Semester x Student

In [ ]:
studentMergeDf = semesterDf.merge(studentDf, how="cross")

In [ ]:
studentMergeDf

### Function Create AKM Student Status

In [ ]:
def statusAkm(row):
    row

### Final AKM

In [ ]:
akmFinalDf = studentDf.apply(statusAkm)